In [1]:
import os
import gc
import random
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score, log_loss
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.decomposition import PCA

import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from scipy.stats import randint, uniform

# Settings
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20

In [2]:
# Dataset 
class SiameseDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform
        self.class_to_indices = {i: np.where(labels == i)[0] for i in np.unique(labels)}

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img1_path = self.file_paths[idx]
        label1 = self.labels[idx]

        if random.random() < 0.5:  # positive pair
            idx2 = random.choice(self.class_to_indices[label1])
            pair_label = 1
        else:  # negative pair
            neg_label = random.choice([l for l in self.class_to_indices.keys() if l != label1])
            idx2 = random.choice(self.class_to_indices[neg_label])
            pair_label = 0

        img2_path = self.file_paths[idx2]

        img1 = Image.open(img1_path).convert("RGB")
        img2 = Image.open(img2_path).convert("RGB")
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, torch.tensor(pair_label, dtype=torch.float32)

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [3]:
# Load Dataset
# Define DATA_PATH and CLASSES
DATA_PATH = "C:/Users/MSC055/Desktop/Sabrina/Eye Image Dataset"
CLASSES = sorted(os.listdir(DATA_PATH))

file_paths, labels = [], []
for idx, cls in enumerate(CLASSES):
    cls_path = os.path.join(DATA_PATH, cls)
    for f in os.listdir(cls_path):
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            file_paths.append(os.path.join(cls_path, f))
            labels.append(idx)

file_paths = np.array(file_paths)
labels = np.array(labels)
# Train/Val/Test split
X_train, X_temp, y_train, y_temp = train_test_split(file_paths, labels, test_size=0.3, stratify=labels, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

print(f"Train_size: {len(X_train)}\nVal_size: {len(X_val)}\nTest_size: {len(X_test)}")

train_dataset = SiameseDataset(X_train, y_train, transform=train_transform)
val_dataset = SiameseDataset(X_val, y_val, transform=val_transform)
test_dataset = SiameseDataset(X_test, y_test, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


Train_size: 3734
Val_size: 800
Test_size: 801


In [4]:
# Siamese Network
class SiameseNetwork(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, x1, x2):
        f1 = self.backbone(x1)
        f2 = self.backbone(x2)
        if len(f1.shape) > 2:
            f1 = torch.flatten(f1, 1)
            f2 = torch.flatten(f2, 1)
        return f1, f2

class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, f1, f2, label):
        distance = (f1 - f2).pow(2).sum(1).sqrt()
        loss = label * distance.pow(2) + (1 - label) * torch.clamp(self.margin - distance, min=0).pow(2)
        return loss.mean()

def extract_embeddings(model, file_paths, transform):
    model.eval()
    embeddings = []
    with torch.no_grad():
        for path in tqdm(file_paths, desc="Extracting Embeddings"):
            img = Image.open(path).convert("RGB")
            img = transform(img).unsqueeze(0).to(DEVICE)
            feat = model.backbone(img)
            if len(feat.shape) > 2:
                feat = torch.flatten(feat, 1)
            embeddings.append(feat.cpu().numpy())
    return np.vstack(embeddings)


In [ ]:
# Backbone Models
backbones = {
    "densenet121": models.densenet121
    #"vgg16": models.vgg16
}

# Symbolic Models
symbolic_models = {
    "RandomForest": (
        RandomForestClassifier(random_state=SEED),
        {
            "n_estimators": randint(50, 100),
            "max_depth": randint(5, 50),
            "min_samples_split": randint(2, 20),
            "min_samples_leaf": randint(1, 10),
            "max_features": ["sqrt", "log2", None],
            "bootstrap": [True, False]
        }
    ),
    "XGBoost": (
        xgb.XGBClassifier(eval_metric="mlogloss", random_state=SEED),
        {
            "n_estimators": randint(50, 100),
            "max_depth": randint(3, 15),
            "learning_rate": uniform(0.01, 0.3),
            "subsample": uniform(0.5, 0.5),
            "colsample_bytree": uniform(0.5, 0.5),
            "gamma": uniform(0, 5),
            "reg_alpha": uniform(0, 1),
            "reg_lambda": uniform(0, 1)
        }
    ),
    
    "LightGBM": (
        lgb.LGBMClassifier(verbose=-1, random_state=SEED),
        {
            "n_estimators": randint(50, 100),
            "max_depth": randint(3, 15),
            "learning_rate": uniform(0.01, 0.3),
            "subsample": uniform(0.5, 0.5),
            "colsample_bytree": uniform(0.5, 0.5),
            "gamma": uniform(0, 5),
            "reg_alpha": uniform(0, 1),
            "reg_lambda": uniform(0, 1)
        }
    ),
    "CatBoost": (
        cb.CatBoostClassifier(verbose=0,random_state=SEED),
        {
            "iterations": randint(50, 100),
            "depth": randint(3, 12),
            "learning_rate": uniform(0.01, 0.3),
            "l2_leaf_reg": uniform(1, 10),
            "border_count": randint(32, 255),
            "random_strength": uniform(0, 10)
        }
    ),
    "AdaBoost": (
        AdaBoostClassifier(random_state=SEED),
        {
           "n_estimators": randint(50, 100),
            "learning_rate": uniform(0.01, 1.0)
        }
    ),
    "GradientBoosting": (
        GradientBoostingClassifier(random_state=SEED),
        {
             "n_estimators": randint(50, 100),
            "learning_rate": uniform(0.01, 0.3),
            "max_depth": randint(3, 15),
            "min_samples_split": randint(2, 20),
            "min_samples_leaf": randint(1, 10),
            "subsample": uniform(0.5, 0.5),
            "max_features": ["sqrt", "log2", None]
        }
    )
}


In [ ]:
# Training Loop
results_summary = []

for backbone_name, backbone_func in backbones.items():
    print(f"\nBackbone: {backbone_name}")
    
    backbone_model = backbone_func(pretrained=True)
    # Remove classifier
    if hasattr(backbone_model, "fc"): backbone_model.fc = nn.Identity()
    if hasattr(backbone_model, "classifier"): backbone_model.classifier = nn.Identity()
    if hasattr(backbone_model, "head"): backbone_model.head = nn.Identity()
    backbone_model = nn.Sequential(*list(backbone_model.children())[:-1])
    backbone_model.to(DEVICE).eval()
    
    siamese_model = SiameseNetwork(backbone_model)
    criterion = ContrastiveLoss()
    optimizer = torch.optim.Adam(siamese_model.parameters(), lr=1e-4)
    
    # Train Siamese
    for epoch in range(EPOCHS):
        siamese_model.train()
        running_loss = 0.0
        for imgs1, imgs2, labels_pair in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            imgs1, imgs2, labels_pair = imgs1.to(DEVICE), imgs2.to(DEVICE), labels_pair.to(DEVICE)
            f1, f2 = siamese_model(imgs1, imgs2)
            loss = criterion(f1, f2, labels_pair)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs1.size(0)
        train_loss = running_loss / len(train_loader.dataset)
    
    # Extract features
    X_train_feat = extract_embeddings(siamese_model, X_train, val_transform)
    X_val_feat = extract_embeddings(siamese_model, X_val, val_transform)
    original_dim = X_train_feat.shape[1]
    print(f"Original feature dimension: {original_dim}")

    for sym_name, (clf, param_dist) in symbolic_models.items():
        print(f"\nSymbolic Model: {sym_name}")
        search = RandomizedSearchCV(clf, param_distributions=param_dist, n_iter=10, cv=3,
                                    scoring="accuracy", random_state=SEED, n_jobs=1)
        search.fit(X_train_feat, y_train)
        tuned_model = search.best_estimator_
        best_params = search.best_params_
        
        y_train_pred = tuned_model.predict(X_train_feat)
        y_val_pred = tuned_model.predict(X_val_feat)
        
        y_train_proba = tuned_model.predict_proba(X_train_feat)
        y_val_proba = tuned_model.predict_proba(X_val_feat)
        
        train_loss_calc = log_loss(y_train, y_train_proba)
        val_loss_calc = log_loss(y_val, y_val_proba)
        
        train_acc = accuracy_score(y_train, y_train_pred)
        val_acc = accuracy_score(y_val, y_val_pred)
        overall_acc = val_acc
        precision = precision_score(y_val, y_val_pred, average='macro', zero_division=0)
        recall = recall_score(y_val, y_val_pred, average='macro', zero_division=0)
        f1 = f1_score(y_val, y_val_pred, average='macro', zero_division=0)
        class_report = classification_report(y_val, y_val_pred, zero_division=0)
        conf_matrix = confusion_matrix(y_val, y_val_pred)

        print(f"\n{backbone_name} + {sym_name}")
        print(f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Train Loss: {train_loss_calc:.4f} | Val Loss: {val_loss_calc:.4f}")
        print(f"Accuracy: {overall_acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1-score: {f1:.4f}")
        print(f"Best Params: {best_params}")
        print("Classification Report:\n", class_report)
        print("Confusion Matrix:\n", conf_matrix)
        
        results_summary.append({
            "backbone": backbone_name,
            "symbolic_model": sym_name,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "train_loss": train_loss_calc,
            "val_loss": val_loss_calc,
            "overall_acc": overall_acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "conf_matrix": conf_matrix,
            "class_report": class_report,
            "best_params": best_params,
            "model": tuned_model,
            "backbone_model": siamese_model
        })

    # Summary 
    print(f"\nCompleted backbone {backbone_name} summary:")
    for r in [res for res in results_summary if res["backbone"] == backbone_name]:
        print(f"{r['symbolic_model']:<15} | Train Acc : {r['train_acc']:.4f} | "
              f"Train Loss : {r['train_loss']:.4f} | Val Acc: {r['val_acc']:.4f} | "
              f"Val Loss: {r['val_loss']:.4f} | Accuracy : {r['overall_acc']:.4f}| "
              f"Precision : {r['precision']:.4f} | Recall : {r['recall']:.4f} | F1: {r['f1']:.4f}")
    

c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Backbone: densenet121


Extracting Embeddings: 100%|██████████| 800/800 [00:32<00:00, 24.99it/s]


Original feature dimension: 50176

Symbolic Model: RandomForest

densenet121 + RandomForest
Train Acc: 0.8781 | Val Acc: 0.5675 | Train Loss: 0.6202 | Val Loss: 1.2119
Accuracy: 0.5675 | Precision: 0.5707 | Recall: 0.3589 | F1-score: 0.3856
Best Params: {'bootstrap': True, 'max_depth': 15, 'max_features': None, 'min_samples_leaf': 8, 'min_samples_split': 5, 'n_estimators': 89}
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        15
           1       0.66      0.94      0.78       226
           2       0.50      0.05      0.10        19
           3       0.48      0.57      0.52       202
           4       0.49      0.53      0.51       153
           5       0.29      0.03      0.05        67
           6       0.62      0.44      0.52        75
           7       1.00      0.67      0.80         3
           8       1.00      0.26      0.42        19
           9       0.67      0.10      0.17        21



c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\MSC055\.conda\envs\pytorch2\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with featur


densenet121 + LightGBM
Train Acc: 0.9344 | Val Acc: 0.6400 | Train Loss: 0.1051 | Val Loss: 1.2863
Accuracy: 0.6400 | Precision: 0.7358 | Recall: 0.5129 | F1-score: 0.5510
Best Params: {'colsample_bytree': np.float64(0.6460723242676091), 'gamma': np.float64(1.8318092164684585), 'learning_rate': np.float64(0.14682099526511078), 'max_depth': 5, 'n_estimators': 93, 'reg_alpha': np.float64(0.5142344384136116), 'reg_lambda': np.float64(0.5924145688620425), 'subsample': np.float64(0.5232252063599989)}
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.07      0.12        15
           1       0.78      0.92      0.85       226
           2       0.67      0.21      0.32        19
           3       0.55      0.65      0.59       202
           4       0.54      0.57      0.56       153
           5       0.60      0.22      0.33        67
           6       0.59      0.57      0.58        75
           7       1.00      1.00      1.

KeyboardInterrupt: 